# Dunnhumby K=1 M4 `q_C` 배정 대조군 — 10 seeds

단일 seed에서 통과한 M4 수식과 판정 구조를 바꾸지 않고 seeds 42~51에서 반복합니다.

- 학습: DAY 1~683
- 개발평가: DAY 684~690
- 과제: 학습기간의 `(user, item)` 쌍을 정답과 후보에서 제외한 신규상품 추천
- 매 seed: M1, 실제 `q_C` M4, degree-matched `q_C` 순열 M4
- 고정: binary graph, `MIN_ITEM_INTER=1`, 100 epoch, 균등 음성 1개, 외부 재정렬 없음
- seed 42 완료 결과는 입력·설정·세 arm checkpoint가 모두 일치할 때만 재사용합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '19ac437282e14a89ec1108d8d57107043cde4b2c'
REPO_DIR = '/content/clv-m2-lightgcn-runner'
!if [ ! -d "$REPO_DIR/.git" ]; then git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git "$REPO_DIR"; fi
!git -C "$REPO_DIR" fetch -q origin $REVIEWED_SHA
!git -C "$REPO_DIR" checkout -q $REVIEWED_SHA
%cd /content/clv-m2-lightgcn-runner
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
import torch
import lightgcn_clv_m4_k1_assignment_control_multiseed as controls

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert controls.CODE_VERSION == 'm4-personalized-positive-weight-k1-assignment-control-multiseed-v1'
cfg = controls.configure_m4_k1_assignment_control_multiseed()
summary = controls.preflight_summary(cfg)
assert cfg.seeds == tuple(range(42, 52))
assert cfg.negative_count == 1
assert summary['trained_models_per_seed'] == list(controls.MODEL_IDS)
assert summary['fixed']['new_item_task'] is True
assert summary['fixed']['min_item_interactions'] == 1
assert summary['fixed']['final_test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
result_df = controls.run_m4_k1_assignment_control_multiseed(cfg)


In [ ]:
import pandas as pd
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

core = [
    'seed', 'model_id', 'm4_assignment', 'row_weight_cv',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10', 'vndcg@10',
    'coverage@10', 'user_value_tendency_recommended_price_alignment',
]
means = pd.DataFrame(result_df.attrs['metric_summary_records'])
paired = pd.DataFrame(result_df.attrs['paired_summary_records'])
comparison = pd.DataFrame(result_df.attrs['comparison_records'])
overlap = pd.DataFrame(result_df.attrs['top10_overlap_records'])

print('1) seed별 M1·실제 M4·q_C 순열 M4 핵심 절대지표')
show(result_df[core])
print('2) 10시드 전체 지표 평균·표준편차')
show(means)
print('3) 두 사전 주지표의 실제 M4 대응차')
show(paired[paired.metric.isin(controls.ECONOMIC_METRICS)])
print('4) seed별 전체 지표 대응차')
show(comparison)
print('5) Top-10 변경 비율')
show(overlap)
print('6) 10시드 사전 고정 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('7) seed별 판독')
print(json.dumps(result_df.attrs['per_seed_readings'], ensure_ascii=False, indent=2))
print('8) q_C 순열 불변식')
print(json.dumps(result_df.attrs['shuffle_diagnostics'], ensure_ascii=False, indent=2))
print('저장 파일:', json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
